# NeuraSight — Chest X-ray Dataset Setup

**Purpose:** Prepare the Chest X-ray dataset (Normal, Pneumonia, Tuberculosis) once and organize it on Google Drive for all future training notebooks.

**Dataset:** [Kaggle - Muhammad Rehan](https://www.kaggle.com/datasets/muhammadrehan00/chest-xray-dataset)

**Idempotent:** Running this notebook multiple times will NOT re-download if the dataset already exists.

## 1. Project Information

In [ ]:
print("=" * 60)
print("  NeuraSight — Chest X-ray Dataset Setup")
print("=" * 60)
print()
print("  Project: NeuraSight - AI-Powered Medical Imaging Platform")
print("  Module:  Chest X-ray Disease Detection")
print("  Classes: Normal, Pneumonia, Tuberculosis")
print("  Source:  Kaggle (muhammadrehan00/chest-xray-dataset)")
print()
print("  This notebook downloads and organizes the dataset on")
print("  Google Drive so future notebooks can load it directly.")
print("=" * 60)

## 2. Install Dependencies

In [ ]:
import subprocess
import importlib

deps = ['kaggle', 'tqdm', 'gdown']
for dep in deps:
    try:
        importlib.import_module(dep)
        print(f"  ✓ {dep} already installed")
    except ImportError:
        print(f"  Installing {dep}...")
        subprocess.run(['pip', 'install', dep, '-q'], check=True)
        print(f"  ✓ {dep} installed")

## 3. Mount Google Drive

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# Define paths
NEURASIGHT_ROOT = "/content/drive/MyDrive/NeuraSight"
DATASET_ROOT = f"{NEURASIGHT_ROOT}/datasets/chest_xray"
RAW_DIR = f"{DATASET_ROOT}/raw"
PROCESSED_DIR = f"{DATASET_ROOT}/processed"
MODELS_DIR = f"{NEURASIGHT_ROOT}/trained_models/chest_xray"
REPORTS_DIR = f"{NEURASIGHT_ROOT}/reports/chest_xray"
NOTEBOOKS_DIR = f"{NEURASIGHT_ROOT}/notebooks/chest_xray"

# Create all directories
dirs = [RAW_DIR, PROCESSED_DIR, MODELS_DIR, REPORTS_DIR, NOTEBOOKS_DIR]
for d in dirs:
    os.makedirs(d, exist_ok=True)

print("\n  Folder structure created:")
print(f"  {NEURASIGHT_ROOT}/")
print(f"  ├── datasets/chest_xray/")
print(f"  │   ├── raw/          (train/val/test splits)")
print(f"  │   └── processed/    (augmented/prepared)")
print(f"  ├── trained_models/chest_xray/")
print(f"  ├── reports/chest_xray/")
print(f"  └── notebooks/chest_xray/")

## 4. Kaggle Authentication

In [ ]:
import os
from pathlib import Path

kaggle_dir = Path("/root/.kaggle")
kaggle_json = kaggle_dir / "kaggle.json"

if kaggle_json.exists():
    print("  ✓ kaggle.json found")
else:
    print("  ⚠ kaggle.json not found. Please upload it.")
    from google.colab import files
    uploaded = files.upload()
    
    kaggle_dir.mkdir(parents=True, exist_ok=True)
    for fn, content in uploaded.items():
        with open(kaggle_json, 'wb') as f:
            f.write(content)
    print(f"  ✓ kaggle.json saved to {kaggle_json}")

# Set permissions
os.chmod(str(kaggle_json), 0o600)

# Verify authentication
try:
    import kaggle
    kaggle.api.authenticate()
    print("  ✓ Kaggle API authenticated successfully")
except Exception as e:
    print(f"  ✗ Authentication failed: {e}")
    raise

## 5. Check Existing Dataset

In [ ]:
# Check if dataset already exists on Drive
train_dir = os.path.join(RAW_DIR, "train")
DATASET_EXISTS = os.path.isdir(train_dir) and len(os.listdir(train_dir)) > 0

if DATASET_EXISTS:
    print("  ✓ Dataset already exists on Drive. Skipping download.")
    print(f"    Location: {RAW_DIR}")
    class_dirs = os.listdir(train_dir)
    print(f"    Classes found: {class_dirs}")
else:
    print("  Dataset not found. Will download in next section.")

## 6. Download Dataset

In [ ]:
if not DATASET_EXISTS:
    print("  Downloading dataset from Kaggle...")
    print("  Dataset: muhammadrehan00/chest-xray-dataset")
    print()
    
    TEMP_DIR = "/content/temp_download"
    os.makedirs(TEMP_DIR, exist_ok=True)
    
    !kaggle datasets download -d muhammadrehan00/chest-xray-dataset -p {TEMP_DIR} --force
    
    print()
    print("  ✓ Download complete")
    zip_files = [f for f in os.listdir(TEMP_DIR) if f.endswith('.zip')]
    for f in zip_files:
        size_mb = os.path.getsize(os.path.join(TEMP_DIR, f)) / (1024*1024)
        print(f"    {f} ({size_mb:.1f} MB)")
else:
    print("  Skipped (dataset already exists)")

## 7. Extract Dataset

In [ ]:
if not DATASET_EXISTS:
    import zipfile
    from tqdm import tqdm
    
    EXTRACT_DIR = "/content/chest_xray_temp"
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    
    zip_path = os.path.join(TEMP_DIR, os.listdir(TEMP_DIR)[0])
    print(f"  Extracting: {os.path.basename(zip_path)}")
    
    with zipfile.ZipFile(zip_path, 'r') as zf:
        members = zf.namelist()
        print(f"  Total files: {len(members)}")
        for member in tqdm(members, desc="  Extracting"):
            zf.extract(member, EXTRACT_DIR)
    
    print("  ✓ Extraction complete")
    print(f"  Contents: {os.listdir(EXTRACT_DIR)}")
    
    # Find the actual data root (might be nested)
    # Look for train/val/test structure
    data_root = EXTRACT_DIR
    for item in os.listdir(EXTRACT_DIR):
        item_path = os.path.join(EXTRACT_DIR, item)
        if os.path.isdir(item_path):
            sub_items = os.listdir(item_path)
            if 'train' in sub_items or 'test' in sub_items:
                data_root = item_path
                break
    
    # Verify expected structure
    expected = ['train', 'val', 'test']
    found = [d for d in expected if os.path.isdir(os.path.join(data_root, d))]
    print(f"  Found splits: {found}")
    
    if 'data.yaml' in os.listdir(data_root):
        print("  Found: data.yaml")
else:
    print("  Skipped (dataset already exists)")

## 8. Copy to Google Drive

In [ ]:
if not DATASET_EXISTS:
    import shutil
    from tqdm import tqdm
    
    print(f"  Copying to Google Drive...")
    print(f"  Source: {data_root}")
    print(f"  Dest:   {RAW_DIR}")
    
    # Copy each split
    for split in ['train', 'val', 'test']:
        src = os.path.join(data_root, split)
        dst = os.path.join(RAW_DIR, split)
        if os.path.isdir(src):
            if os.path.exists(dst):
                shutil.rmtree(dst)
            shutil.copytree(src, dst)
            n_files = sum(len(files) for _, _, files in os.walk(dst))
            print(f"    ✓ {split}: {n_files} files copied")
    
    # Copy data.yaml if exists
    yaml_src = os.path.join(data_root, 'data.yaml')
    if os.path.exists(yaml_src):
        shutil.copy2(yaml_src, os.path.join(RAW_DIR, 'data.yaml'))
        print("    ✓ data.yaml copied")
    
    # Cleanup temp files
    print("  Cleaning up temporary files...")
    shutil.rmtree(TEMP_DIR, ignore_errors=True)
    shutil.rmtree(EXTRACT_DIR, ignore_errors=True)
    print("  ✓ Temporary files removed")
    print()
    print("  ✓ Dataset successfully copied to Google Drive!")
else:
    print("  Skipped (dataset already exists)")

## 9. Dataset Verification

In [ ]:
print("  DATASET STRUCTURE")
print("  " + "=" * 50)
print(f"  Location: {RAW_DIR}")
print()

splits = ['train', 'val', 'test']
split_data = {}

for split in splits:
    split_dir = os.path.join(RAW_DIR, split)
    if not os.path.isdir(split_dir):
        continue
    classes = sorted(os.listdir(split_dir))
    split_data[split] = {}
    for cls in classes:
        cls_dir = os.path.join(split_dir, cls)
        if os.path.isdir(cls_dir):
            count = len([f for f in os.listdir(cls_dir) 
                        if f.lower().endswith(('.jpg','.jpeg','.png'))])
            split_data[split][cls] = count

for split, class_counts in split_data.items():
    print(f"  {split.upper()}/")
    for cls, count in class_counts.items():
        print(f"    {cls:15s}: {count:,} images")
    total_label = 'TOTAL'
    print(f"    {total_label:15s}: {sum(class_counts.values()):,}")
    print()

total = sum(sum(cc.values()) for cc in split_data.values())
print(f"  GRAND TOTAL: {total:,} images")

## 10. Dataset Summary

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Build summary dataframe
rows = []
for split, class_counts in split_data.items():
    for cls, count in class_counts.items():
        rows.append({'Split': split, 'Class': cls, 'Images': count})

df = pd.DataFrame(rows)
print("  Dataset Summary:")
print(df.pivot_table(index='Class', columns='Split', values='Images', 
                     aggfunc='sum', margins=True, margins_name='Total'))

# Bar charts
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
colors = {'normal': '#2ecc71', 'pneumonia': '#e74c3c', 'tuberculosis': '#3498db'}

for idx, split in enumerate(splits):
    if split in split_data:
        classes = list(split_data[split].keys())
        counts = list(split_data[split].values())
        bars = axes[idx].bar(classes, counts, 
                            color=[colors.get(c, '#95a5a6') for c in classes])
        axes[idx].set_title(f'{split.upper()} Distribution')
        axes[idx].set_ylabel('Images')
        for bar, count in zip(bars, counts):
            axes[idx].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 5,
                          str(count), ha='center', fontweight='bold', fontsize=9)

# Overall
all_classes = sorted(set(c for cc in split_data.values() for c in cc))
overall = {c: sum(split_data[s].get(c, 0) for s in splits) for c in all_classes}
bars = axes[3].bar(overall.keys(), overall.values(),
                   color=[colors.get(c, '#95a5a6') for c in overall.keys()])
axes[3].set_title('OVERALL Distribution')
axes[3].set_ylabel('Images')
for bar, count in zip(bars, overall.values()):
    axes[3].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 5,
                str(count), ha='center', fontweight='bold', fontsize=9)

plt.suptitle('Chest X-ray Dataset Distribution', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(REPORTS_DIR, 'dataset_distribution.png'), 
            dpi=150, bbox_inches='tight')
plt.show()

## 11. Save Metadata

In [ ]:
import json
from datetime import datetime

dataset_info = {
    "dataset_name": "Chest X-ray (Normal, Pneumonia, Tuberculosis)",
    "source": "https://www.kaggle.com/datasets/muhammadrehan00/chest-xray-dataset",
    "download_date": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "total_images": total,
    "classes": list(all_classes),
    "num_classes": len(all_classes),
    "splits": {
        split: {
            "total": sum(class_counts.values()),
            "classes": class_counts
        }
        for split, class_counts in split_data.items()
    },
    "dataset_path": RAW_DIR,
    "image_formats": ["jpg", "jpeg", "png"],
    "project": "NeuraSight",
    "module": "chest_xray"
}

info_path = os.path.join(DATASET_ROOT, "dataset_info.json")
with open(info_path, 'w') as f:
    json.dump(dataset_info, f, indent=2)

print(f"  ✓ Metadata saved: {info_path}")
print()
print("  Contents:")
print(json.dumps(dataset_info, indent=2))

## 12. Completion

In [ ]:
print()
print("=" * 60)
print("  ✓ DATASET SUCCESSFULLY PREPARED")
print("=" * 60)
print()
print("  Future notebooks can load directly from:")
print(f"  {RAW_DIR}/")
print()
print("  Structure:")
print(f"    train/  → Training data")
print(f"    val/    → Validation data")
print(f"    test/   → Test data")
print()
print("  Classes: normal, pneumonia, tuberculosis")
print()
print("  Next step: Run 'Chest_Xray_Model_Training.ipynb'")
print("=" * 60)